# Notebook 4: Query & Retrieval Demo

**Goal**: Explore and compare all retrieval strategies.

## Topics:
- Dense retrieval
- BM25 sparse retrieval
- Hybrid (RRF) retrieval
- Cross-encoder reranking
- Query transformation (HyDE, MultiQuery)
- MMR diversity


In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_pipeline.document_loader import DocumentLoader
from data_pipeline.chunking import ChunkingEngine
from embeddings.embedding import EmbeddingEngine
from vector_store.vector_store import FAISSVectorStore
from retrieval.retriever import DenseRetriever, BM25Retriever, HybridRetriever, maximal_marginal_relevance
from retrieval.reranker import CrossEncoderReranker

print('Setup complete')

## 4.1 Build Index

In [ ]:
loader = DocumentLoader()
docs = loader.load('../sample_data/', loader_type='directory')

chunker = ChunkingEngine(strategy='recursive', chunk_size=200, chunk_overlap=30)
chunks = chunker.split(docs)

engine = EmbeddingEngine(
    provider='sentence_transformers',
    model_name='all-MiniLM-L6-v2',
    use_cache=True,
    cache_dir='../.embed_cache',
)

texts = [c.content for c in chunks]
result = engine.encode(texts)
vectors = result.embeddings
ids = [c.chunk_id for c in chunks]
metadatas = [c.metadata for c in chunks]

# Vector store
store = FAISSVectorStore(dimension=engine.dimension, index_type='flat', metric='cosine')
store.add(ids, vectors, texts, metadatas)

# Dense retriever
dense = DenseRetriever(store, engine)

# BM25 retriever
bm25 = BM25Retriever()
bm25.build(texts, ids, metadatas)

# Hybrid retriever
hybrid = HybridRetriever(dense, bm25, use_rrf=True)

print(f'Index ready: {store.count} vectors')

## 4.2 Compare Retrieval Strategies Side-by-Side

In [ ]:
def show_results(query, results, title, n=3):
    print(f'\n{"="*60}')
    print(f'{title}')
    print(f'Query: "{query}"')
    print('='*60)
    for i, r in enumerate(results[:n]):
        print(f'[{i+1}] Score: {r.score:.4f} | Source: {r.metadata.get("file_type","?")}')
        print(f'    {r.content[:120]}')
        print()

QUERY = 'When can I get a refund?'

# Dense
dense_results = dense.retrieve(QUERY, top_k=5)
show_results(QUERY, dense_results, '--- DENSE (Semantic) ---')

# BM25
bm25_results = bm25.retrieve(QUERY, top_k=5)
show_results(QUERY, bm25_results, '--- BM25 (Keyword) ---')

# Hybrid
hybrid_results = hybrid.retrieve(QUERY, top_k=5)
show_results(QUERY, hybrid_results, '--- HYBRID (RRF) ---')

## 4.3 Keyword vs Semantic Gap Demo

In [ ]:
# Query that uses DIFFERENT words than document but same meaning
semantic_query = 'merchandise return timeframe'   # synonymous with 'refund within 30 days'

# BM25 will miss this (no matching keywords), dense will find it
dense_semantic = dense.retrieve(semantic_query, top_k=3)
bm25_semantic  = bm25.retrieve(semantic_query, top_k=3)

print(f'Semantic query: "{semantic_query}"')
print(f'\nDense top result  → {dense_semantic[0].content[:100] if dense_semantic else "None"}')
print(f'BM25 top result   → {bm25_semantic[0].content[:100] if bm25_semantic else "None (score too low)"}')

# Now a keyword-specific query (rare term)
keyword_query = 'SEER rating'

dense_kw = dense.retrieve(keyword_query, top_k=3)
bm25_kw  = bm25.retrieve(keyword_query, top_k=3)

print(f'\nKeyword query: "{keyword_query}"')
print(f'Dense top result → {dense_kw[0].content[:100] if dense_kw else "None"}')
print(f'BM25 top result  → {bm25_kw[0].content[:100] if bm25_kw else "None"}')

## 4.4 Cross-Encoder Reranking

In [ ]:
reranker = CrossEncoderReranker(
    model_name='cross-encoder/ms-marco-MiniLM-L-6-v2',
    device='cpu',
    batch_size=16,
)

QUERY = 'How long do refunds take to process?'

# Get broad candidates
candidates = hybrid.retrieve(QUERY, top_k=8)

# Rerank
import time
t0 = time.perf_counter()
reranked = reranker.rerank(QUERY, candidates, top_k=5)
rerank_ms = (time.perf_counter() - t0) * 1000

print(f'Before reranking ({len(candidates)} candidates):')
for r in candidates[:5]:
    print(f'  [{r.score:.4f}] {r.content[:90]}')

print(f'\nAfter cross-encoder reranking (top 5, took {rerank_ms:.1f}ms):')
for r in reranked:
    print(f'  [{r.score:.4f}] {r.content[:90]}')

## 4.5 Retrieval Precision Analysis

In [ ]:
# Manual relevance labels for demo
test_cases = [
    {'query': 'refund time', 'relevant_keywords': ['5-7 business days', 'processed within']},
    {'query': 'laptop specifications', 'relevant_keywords': ['15-inch', '32GB', 'XR-500']},
    {'query': 'international shipping time', 'relevant_keywords': ['10-15 business days', '50 countries']},
]

results_df = []
for tc in test_cases:
    q = tc['query']
    kw = tc['relevant_keywords']
    
    for strategy, retriever_fn in [('Dense', dense.retrieve), ('BM25', bm25.retrieve), ('Hybrid', hybrid.retrieve)]:
        results = retriever_fn(q, top_k=5)
        
        # Check if top-3 contain any relevant keyword
        top3_content = ' '.join(r.content.lower() for r in results[:3])
        hit = any(k.lower() in top3_content for k in kw)
        
        results_df.append({'query': q[:30], 'strategy': strategy, 'hit@3': int(hit)})

df = pd.DataFrame(results_df)
pivot = df.pivot_table(index='query', columns='strategy', values='hit@3')
print('Hit@3 Results (1=found relevant chunk, 0=missed):')
print(pivot.to_string())

print(f'\nAggregate hit@3 per strategy:')
print(df.groupby('strategy')['hit@3'].mean())

## 4.6 MMR: Relevance vs Diversity

In [ ]:
QUERY = 'product return'
qvec = engine.encode_query(QUERY)
candidates = dense.retrieve(QUERY, top_k=8)

# Get embeddings for candidates
cand_texts = [c.content for c in candidates]
cand_vecs = engine.encode(cand_texts).embeddings

# MMR with high lambda (more relevance)
mmr_relevant = maximal_marginal_relevance(
    qvec, cand_vecs, candidates, top_k=4, lambda_param=0.9
)

# MMR with low lambda (more diversity)
mmr_diverse = maximal_marginal_relevance(
    qvec, cand_vecs, candidates, top_k=4, lambda_param=0.3
)

print(f'Query: "{QUERY}"')
print(f'\nHigh Relevance (λ=0.9) — similar chunks grouped together:')
for r in mmr_relevant:
    print(f'  {r.content[:100]}')

print(f'\nHigh Diversity (λ=0.3) — spread across topics:')
for r in mmr_diverse:
    print(f'  {r.content[:100]}')

## 4.7 Retrieval Strategy Decision Framework

| Scenario | Recommended Strategy |
|---|---|
| General QA on documents | Hybrid (RRF) + Cross-encoder rerank |
| Code search / exact terms | BM25 or Hybrid with high BM25 weight |
| Multilingual queries | Dense with multilingual model (bge-m3) |
| Diverse answer coverage | Dense + MMR (λ=0.5) |
| Low latency (<50ms) | Dense only, no rerank |
| Max accuracy, offline | Hybrid + Cross-encoder + MMR |
| Complex multi-part Q | Multi-hop (Notebook 5) |

**Key insight**: Hybrid + rerank gives best recall@10 but adds ~50-100ms. For latency-sensitive applications, dense-only with a fast HNSW index (p50 < 10ms) is often the right trade-off.